# Drone 3‑D State Estimation in Turbulent Wind
### Linear Dynamical Systems + Kalman Filtering — ML Unit 4

**Team Cash Guard** — V. Anbu Chelvan · S. Roopesh · B. Charith Reddy · Y. Rupesh Reddy

---

**Goal.** Recover a drone's true 3‑D position *and* velocity *and* the wind acting on it, from
three individually inadequate sensors: a drifting IMU, a noisy altimeter and an
altitude‑dependent optical‑flow sensor.

**Unit 4 mapping.** An LDS is an HMM with a continuous hidden state. The Kalman filter is the
forward (α) recursion; the RTS smoother is the backward (β) half of forward–backward; the
log‑likelihood accumulated from innovations does the job of scaling factors; EM learns unknown
noise exactly as Baum–Welch learns HMM parameters.

**Real data.** Cell 2 downloads a **real PX4 flight‑controller log** and measures the IMU's true
noise and bias from it. Those measured numbers parameterise the filter's process noise `Q`.

**Real database.** Everything flows through SQLite: the simulator *writes* the sensor stream,
the filter *reads it back with SQL*, estimates are written in, and accuracy is computed by a
SQL `JOIN`. Nothing is handed between stages in memory.

> **Stated plainly:** the 60‑second turbulent trajectory is *simulated*. The IMU noise driving it is
> *measured from real hardware*. The public PX4 logs are bench tests where the aircraft never
> moved, and the ETH EuRoC server did not respond, so a fully real flight was not available.
> Measuring sensor noise from a static log is standard practice — it is how Allan‑variance
> calibration is done.

## 1 · Setup

In [ ]:
!pip -q install pyulog
import numpy as np, sqlite3, json, math, os, urllib.request, datetime
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": .3})
print("ready")

## 2 · Fetch the REAL flight log and measure the IMU

Source (BSD‑3‑Clause): `https://raw.githubusercontent.com/PX4/pyulog/main/test/sample.ulg`

A stationary log is the *right* data for characterising sensor noise: with the vehicle still,
everything the accelerometer reports beyond gravity is noise and bias.

In [ ]:
IMU_LOG_URL = "https://raw.githubusercontent.com/PX4/pyulog/main/test/sample.ulg"
LOG = "sample.ulg"
if not os.path.exists(LOG):
    urllib.request.urlretrieve(IMU_LOG_URL, LOG)
print(f"downloaded {os.path.getsize(LOG)/1e6:.2f} MB")

from pyulog import ULog
_u  = ULog(LOG)
_sc = next(d for d in _u.data_list if d.name == "sensor_combined").data
ACC = np.stack([_sc[f"accelerometer_m_s2[{i}]"] for i in range(3)], 1)
GYR = np.stack([_sc[f"gyro_rad[{i}]"]          for i in range(3)], 1)
TT  = _sc["timestamp"] / 1e6
FS  = len(TT) / (TT[-1] - TT[0])

ACC_SIGMA = np.std(ACC, 0)
GYR_SIGMA = np.std(GYR, 0)
ACC_BIAS  = np.mean(ACC, 0) + np.array([0, 0, 9.80665])   # NED: a still accel reads -g
SIGMA_IMU = float(ACC_SIGMA.mean())
VRW       = float(ACC_SIGMA.mean() / np.sqrt(FS))

print(f"  rate            {FS:.0f} Hz over {TT[-1]-TT[0]:.0f} s, {len(TT)} samples")
print(f"  accel sigma     {ACC_SIGMA.round(3)} m/s^2")
print(f"  gyro  sigma     {GYR_SIGMA.round(4)} rad/s")
print(f"  accel bias      {ACC_BIAS.round(3)} m/s^2   <- real, uncorrected")
print(f"  sigma_imu -> Q  {SIGMA_IMU:.3f} m/s^2   (MEASURED, not assumed)")
print(f"  velocity random walk {VRW:.5f} (m/s^2)/sqrt(Hz)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.1))
for i, lbl in enumerate("xyz"):
    ax[0].plot(TT - TT[0], ACC[:, i], lw=.4, label=f"a{lbl}")
ax[0].set(title="Real accelerometer (PX4 log)", xlabel="s", ylabel="m/s$^2$"); ax[0].legend(fontsize=7)
ax[1].hist(ACC[:, 0] - ACC[:, 0].mean(), bins=90, alpha=.8)
ax[1].set(title=f"Noise is Gaussian  ($\sigma_x$={ACC_SIGMA[0]:.3f})", xlabel="m/s$^2$")
ax[2].bar("xyz", ACC_SIGMA); ax[2].set(title="Per-axis $\sigma$ feeding Q", ylabel="m/s$^2$")
plt.suptitle("FIGURE 1 — the real sensor behind the model", y=1.04, fontsize=11)
plt.tight_layout(); plt.show()

## 3 · The linear dynamical system

Per axis the state is `[position, velocity, wind]`, and all three equations are **linear** —
which is precisely why an exact Kalman filter applies and no EKF is needed:

$$p_{t+1}=p_t+v_t\,\Delta t \qquad
v_{t+1}=v_t+\big(a-k(v_t-w_t)\big)\Delta t \qquad
w_{t+1}=\phi\,w_t+\varepsilon,\;\; \phi=e^{-\Delta t/\tau}$$

Drag acts on **airspeed** $v-w$, not ground speed. That single physical fact couples the wind
into the dynamics and makes it observable from its effect alone.

In [ ]:
AX = 3                      # x, y, z
NS = 3 * AX                 # 9 states: 3 position, 3 velocity, 3 wind


# ----------------------------------------------------------------- model

def build_model(dt=0.01, k_drag=0.8, tau=5.0, sigma_gust=1.5, sigma_imu=0.15):
    """Return A, B, Q for the augmented linear dynamical system."""
    phi = np.exp(-dt / tau)
    # steady-state wind variance is sigma_gust^2, so the driving noise must be:
    q_w = sigma_gust ** 2 * (1 - phi ** 2)

    A = np.zeros((NS, NS))
    B = np.zeros((NS, AX))
    Q = np.zeros((NS, NS))
    for i in range(AX):
        p, v, w = i, AX + i, 2 * AX + i
        A[p, p], A[p, v] = 1.0, dt
        A[v, v], A[v, w] = 1.0 - k_drag * dt, k_drag * dt
        A[w, w] = phi
        B[p, i], B[v, i] = 0.5 * dt ** 2, dt
        # IMU accelerometer noise enters as control noise -> velocity/position
        Q[p, p] = (0.5 * dt ** 2 * sigma_imu) ** 2
        Q[v, v] = (dt * sigma_imu) ** 2
        Q[p, v] = Q[v, p] = (0.5 * dt ** 2 * sigma_imu) * (dt * sigma_imu)
        Q[w, w] = q_w
    return A, B, Q


# Measurement models. Each sensor sees a different part of the state, and none
# of them sees px or py - that is the whole point of the observability finding.
H_ALT = np.zeros((1, NS)); H_ALT[0, 2] = 1.0            # altimeter -> pz
H_FLOW = np.zeros((2, NS)); H_FLOW[0, 3] = H_FLOW[1, 4] = 1.0   # optical flow -> vx, vy
H_GPS = np.zeros((2, NS)); H_GPS[0, 0] = H_GPS[1, 1] = 1.0      # (extension) GPS -> px, py


def commanded_accel(t):
    """What the flight controller asks for: a climbing, turning path."""
    return np.array([
        1.2 * np.cos(0.35 * t),
        1.2 * np.sin(0.35 * t) + 0.4 * np.sin(0.11 * t),
        0.25 * np.sin(0.18 * t),
    ])


# ------------------------------------------------------------- simulation

def simulate(T=60.0, dt=0.01, seed=7, sigma_imu=0.15, sigma_alt=0.60,
             sigma_flow=0.12, f_alt=10, f_flow=30, f_gps=0, sigma_gps=2.5,
             **model_kw):
    """Generate ground truth and noisy, multi-rate sensor readings."""
    rng = np.random.default_rng(seed)
    A, B, Q = build_model(dt=dt, sigma_imu=sigma_imu, **model_kw)
    n = int(T / dt)
    t = np.arange(n) * dt

    phi = A[2 * AX, 2 * AX]
    q_w = Q[2 * AX, 2 * AX]

    X = np.zeros((n, NS))
    X[0, 2] = 5.0                       # start 5 m up
    X[0, 2 * AX:] = rng.normal(0, np.sqrt(q_w / (1 - phi ** 2)), AX)

    u_true = np.zeros((n, AX))
    imu = np.zeros((n, AX))
    for i in range(n - 1):
        a = commanded_accel(t[i])
        u_true[i] = a
        x = X[i]
        p, v, w = x[:AX], x[AX:2 * AX], x[2 * AX:]
        X[i + 1, :AX] = p + v * dt
        X[i + 1, AX:2 * AX] = v + (a - 0.8 * (v - w)) * dt
        X[i + 1, 2 * AX:] = phi * w + rng.normal(0, np.sqrt(q_w), AX)
    u_true[-1] = commanded_accel(t[-1])
    # the IMU reports commanded acceleration corrupted by noise: it is the INPUT u
    imu = u_true + rng.normal(0, sigma_imu, (n, AX))

    # multi-rate sensors; None where that sensor did not fire on this step
    step_alt = max(1, int(round(1 / (f_alt * dt))))
    step_flow = max(1, int(round(1 / (f_flow * dt))))
    step_gps = max(1, int(round(1 / (f_gps * dt)))) if f_gps else 0

    z_alt = [None] * n; z_flow = [None] * n; z_gps = [None] * n
    for i in range(n):
        if i % step_alt == 0:
            z_alt[i] = X[i, 2] + rng.normal(0, sigma_alt)
        if i % step_flow == 0:
            # optical flow gets noisier the higher you fly (fewer ground features)
            s = sigma_flow * (1 + 0.06 * max(0.0, X[i, 2] - 5.0))
            z_flow[i] = X[i, AX:AX + 2] + rng.normal(0, s, 2)
        if step_gps and i % step_gps == 0:
            z_gps[i] = X[i, :2] + rng.normal(0, sigma_gps, 2)

    return dict(t=t, X=X, imu=imu, u_true=u_true, z_alt=z_alt, z_flow=z_flow,
                z_gps=z_gps, A=A, B=B, Q=Q, dt=dt,
                sigma_alt=sigma_alt, sigma_flow=sigma_flow, sigma_gps=sigma_gps)


# ------------------------------------------------------- Kalman filtering

def flow_scale(alt):
    """Optical flow sees fewer ground features the higher you are."""
    return 1.0 + 0.06 * max(0.0, alt - 5.0)


def kalman_filter(sim, R_alt=None, R_flow=None, R_gps=None, Q=None,
                  use_gps=False, collect=True, adaptive_flow=False):
    """
    Forward pass. This is the forward (alpha) recursion of Unit 4, written for
    continuous Gaussian states: predict = sum over previous states, update = Bayes.
    """
    A, B = sim["A"], sim["B"]
    Q = sim["Q"] if Q is None else Q
    n = len(sim["t"])
    R_alt = np.array([[sim["sigma_alt"] ** 2]]) if R_alt is None else R_alt
    R_flow = np.eye(2) * sim["sigma_flow"] ** 2 if R_flow is None else R_flow
    R_gps = np.eye(2) * sim["sigma_gps"] ** 2 if R_gps is None else R_gps

    x = np.zeros(NS); x[2] = 5.0
    P = np.eye(NS) * 1.0
    P[2 * AX:, 2 * AX:] = np.eye(AX) * 2.0 ** 2

    xf = np.zeros((n, NS)); Pf = np.zeros((n, NS, NS))
    xp = np.zeros((n, NS)); Pp = np.zeros((n, NS, NS))
    loglik = 0.0
    gains, nis = [], []

    for i in range(n):
        if i > 0:                                   # ---- predict
            x = A @ x + B @ sim["imu"][i - 1]
            P = A @ P @ A.T + Q
        xp[i], Pp[i] = x, P

        for z, H, R, tag in ((sim["z_alt"][i], H_ALT, R_alt, "alt"),
                             (sim["z_flow"][i], H_FLOW, R_flow, "flow"),
                             (sim["z_gps"][i] if use_gps else None, H_GPS, R_gps, "gps")):
            if z is None:
                continue
            z = np.atleast_1d(z)
            if tag == "flow" and adaptive_flow:
                # scale R by the CURRENT ESTIMATED altitude - we never use truth
                R = R_flow * flow_scale(x[2]) ** 2
            y = z - H @ x                           # innovation
            S = H @ P @ H.T + R                     # innovation covariance
            K = P @ H.T @ np.linalg.inv(S)          # Kalman gain
            x = x + K @ y
            P = (np.eye(NS) - K @ H) @ P
            P = 0.5 * (P + P.T)                     # keep it symmetric
            # log-likelihood: the continuous analogue of Unit 4's scaling factors
            sign, logdet = np.linalg.slogdet(2 * np.pi * S)
            loglik += -0.5 * (logdet + y @ np.linalg.inv(S) @ y)
            if collect:
                if tag == "alt":
                    gains.append(K[2, 0])
                nis.append((i, tag, float(y @ np.linalg.inv(S) @ y), len(z)))
        xf[i], Pf[i] = x, P

    return dict(xf=xf, Pf=Pf, xp=xp, Pp=Pp, loglik=loglik,
                gains=np.array(gains), nis=nis, Q=Q,
                R_alt=R_alt, R_flow=R_flow)


def rts_smoother(sim, filt):
    """
    Backward pass: the Rauch-Tung-Striebel smoother. This is exactly the
    backward (beta) half of forward-backward, for continuous Gaussian states.
    """
    A = sim["A"]
    xf, Pf, xp, Pp = filt["xf"], filt["Pf"], filt["xp"], filt["Pp"]
    n = len(xf)
    xs = xf.copy(); Ps = Pf.copy()
    Js = np.zeros((n, NS, NS))
    for i in range(n - 2, -1, -1):
        J = Pf[i] @ A.T @ np.linalg.inv(Pp[i + 1])
        Js[i] = J
        xs[i] = xf[i] + J @ (xs[i + 1] - xp[i + 1])
        Ps[i] = Pf[i] + J @ (Ps[i + 1] - Pp[i + 1]) @ J.T
        Ps[i] = 0.5 * (Ps[i] + Ps[i].T)
    return dict(xs=xs, Ps=Ps, J=Js)


# ---------------------------------------------------------------- EM (M-step)

def em_learn_noise(sim, iters=12, verbose=True):
    """
    EM for the unknown measurement noises, exactly the Unit 4 recipe:
      E-step: run filter + smoother to get the posterior over hidden states.
      M-step: set each variance to the mean squared smoothed residual.
    We start deliberately wrong (10x too large) and watch it find the truth.
    """
    R_alt = np.array([[(sim["sigma_alt"] * 3.0) ** 2]])
    R_flow = np.eye(2) * (sim["sigma_flow"] * 3.0) ** 2
    hist = []
    for it in range(iters):
        filt = kalman_filter(sim, R_alt=R_alt, R_flow=R_flow, collect=False)
        sm = rts_smoother(sim, filt)
        xs = sm["xs"]; Ps = sm["Ps"]

        # M-step: E[(z - Hx)^2] = residual^2 + H P H^T
        num_a, den_a = 0.0, 0
        num_f, den_f = 0.0, 0
        for i in range(len(xs)):
            if sim["z_alt"][i] is not None:
                r = sim["z_alt"][i] - H_ALT @ xs[i]
                num_a += float(np.squeeze(r) ** 2 + np.squeeze(H_ALT @ Ps[i] @ H_ALT.T)); den_a += 1
            if sim["z_flow"][i] is not None:
                r = np.atleast_1d(sim["z_flow"][i]) - H_FLOW @ xs[i]
                num_f += float(r @ r + np.trace(H_FLOW @ Ps[i] @ H_FLOW.T)); den_f += 2
        R_alt = np.array([[num_a / den_a]])
        R_flow = np.eye(2) * (num_f / den_f)
        hist.append((it, filt["loglik"], float(np.sqrt(R_alt[0, 0])),
                     float(np.sqrt(R_flow[0, 0]))))
        if verbose:
            print(f"   EM {it:2d}:  loglik={filt['loglik']:12.1f}   "
                  f"sigma_alt={np.sqrt(R_alt[0,0]):.3f} (true {sim['sigma_alt']:.3f})   "
                  f"sigma_flow={np.sqrt(R_flow[0,0]):.3f} (true {sim['sigma_flow']:.3f})")
    return R_alt, R_flow, hist


def dead_reckon(sim):
    """IMU-only integration: no filtering. The baseline we must beat."""
    A, B = sim["A"], sim["B"]
    n = len(sim["t"])
    x = np.zeros(NS); x[2] = 5.0
    out = np.zeros((n, NS))
    for i in range(n):
        if i > 0:
            x = A @ x + B @ sim["imu"][i - 1]
        out[i] = x
    return out


def rmse(a, b):
    return float(np.sqrt(np.mean(np.sum((a - b) ** 2, axis=1))))

## 4 · The database

Six tables. The filter's **only** input is `sensor_readings`, read back out with SQL.

In [ ]:
SCHEMA = """
PRAGMA journal_mode=WAL;
CREATE TABLE IF NOT EXISTS runs(
    run_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    created     TEXT NOT NULL,
    params      TEXT NOT NULL,
    pos_rmse_dr REAL, pos_rmse_kf REAL, pos_rmse_rts REAL,
    vel_rmse_kf REAL, wind_rmse_kf REAL, loglik REAL,
    nees REAL, nis_alt REAL, nis_flow REAL, notes TEXT);

CREATE TABLE IF NOT EXISTS ground_truth(
    run_id INTEGER NOT NULL, step INTEGER NOT NULL, t REAL NOT NULL,
    px REAL, py REAL, pz REAL, vx REAL, vy REAL, vz REAL,
    wx REAL, wy REAL, wz REAL,
    PRIMARY KEY(run_id, step));

CREATE TABLE IF NOT EXISTS sensor_readings(
    run_id INTEGER NOT NULL, step INTEGER NOT NULL, t REAL NOT NULL,
    sensor TEXT NOT NULL,            -- 'imu' | 'altimeter' | 'optical_flow' | 'gps'
    c0 REAL, c1 REAL, c2 REAL,       -- up to 3 channels
    sigma REAL,                      -- the sensor's stated 1-sigma at this instant
    PRIMARY KEY(run_id, step, sensor));
CREATE INDEX IF NOT EXISTS idx_sensor ON sensor_readings(run_id, sensor, step);

CREATE TABLE IF NOT EXISTS imu_calibration(
    source      TEXT PRIMARY KEY,   -- URL the log was fetched from
    fetched     TEXT NOT NULL,
    rate_hz     REAL, duration_s REAL, n_samples INTEGER,
    acc_sigma_x REAL, acc_sigma_y REAL, acc_sigma_z REAL,
    gyr_sigma_x REAL, gyr_sigma_y REAL, gyr_sigma_z REAL,
    acc_bias_x  REAL, acc_bias_y  REAL, acc_bias_z  REAL,
    vrw         REAL);

CREATE TABLE IF NOT EXISTS imu_raw(          -- real samples, straight off the aircraft
    source TEXT NOT NULL, n INTEGER NOT NULL, t REAL,
    ax REAL, ay REAL, az REAL, gx REAL, gy REAL, gz REAL,
    PRIMARY KEY(source, n));

CREATE TABLE IF NOT EXISTS estimates(
    run_id INTEGER NOT NULL, step INTEGER NOT NULL, t REAL NOT NULL,
    kind TEXT NOT NULL,              -- 'filter' | 'smoother' | 'dead_reckon'
    px REAL, py REAL, pz REAL, vx REAL, vy REAL, vz REAL,
    wx REAL, wy REAL, wz REAL,
    var_px REAL, var_py REAL, var_pz REAL,
    PRIMARY KEY(run_id, step, kind));
"""


def connect(path="drone.db"):
    con = sqlite3.connect(path, timeout=30, isolation_level=None)  # autocommit
    con.row_factory = sqlite3.Row
    con.executescript(SCHEMA)
    return con


def new_run(con, params, notes=""):
    cur = con.execute(
        "INSERT INTO runs(created, params, notes) VALUES(datetime('now'), ?, ?)",
        (json.dumps(params), notes))
    return cur.lastrowid


def write_truth(con, run_id, t, X):
    con.executemany(
        "INSERT OR REPLACE INTO ground_truth VALUES(?,?,?,?,?,?,?,?,?,?,?,?)",
        [(run_id, i, float(t[i]), *[float(v) for v in X[i]]) for i in range(len(X))])


def write_sensors(con, run_id, t, imu, z_alt, z_flow, z_gps, s_alt, s_flow_list, s_gps):
    rows = []
    for i in range(len(t)):
        rows.append((run_id, i, float(t[i]), "imu",
                     float(imu[i][0]), float(imu[i][1]), float(imu[i][2]), None))
        if z_alt[i] is not None:
            rows.append((run_id, i, float(t[i]), "altimeter",
                         float(z_alt[i]), None, None, s_alt))
        if z_flow[i] is not None:
            rows.append((run_id, i, float(t[i]), "optical_flow",
                         float(z_flow[i][0]), float(z_flow[i][1]), None,
                         float(s_flow_list[i])))
        if z_gps[i] is not None:
            rows.append((run_id, i, float(t[i]), "gps",
                         float(z_gps[i][0]), float(z_gps[i][1]), None, s_gps))
    con.executemany("INSERT OR REPLACE INTO sensor_readings VALUES(?,?,?,?,?,?,?,?)", rows)


def read_sensor_stream(con, run_id, n):
    """
    The filter's ONLY input. Pull the whole multi-rate stream back out with SQL and
    re-assemble it into per-step slots. Steps with no reading simply have none.
    """
    imu = np.zeros((n, 3))
    z_alt = [None] * n
    z_flow = [None] * n
    z_gps = [None] * n
    s_flow = [None] * n
    for r in con.execute(
            "SELECT step, sensor, c0, c1, c2, sigma FROM sensor_readings "
            "WHERE run_id=? ORDER BY step", (run_id,)):
        i, s = r["step"], r["sensor"]
        if s == "imu":
            imu[i] = (r["c0"], r["c1"], r["c2"])
        elif s == "altimeter":
            z_alt[i] = r["c0"]
        elif s == "optical_flow":
            z_flow[i] = np.array([r["c0"], r["c1"]]); s_flow[i] = r["sigma"]
        elif s == "gps":
            z_gps[i] = np.array([r["c0"], r["c1"]])
    return imu, z_alt, z_flow, z_gps, s_flow


def write_estimates(con, run_id, t, kind, X, P=None):
    rows = []
    for i in range(len(X)):
        v = [float(x) for x in X[i]]
        if P is None:
            var = (None, None, None)
        else:
            var = tuple(float(P[i][j, j]) for j in range(3))
        rows.append((run_id, i, float(t[i]), kind, *v, *var))
    con.executemany(
        "INSERT OR REPLACE INTO estimates VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", rows)


def rmse_by_sql(con, run_id, kind):
    """Compute the error in the database itself, with a JOIN - no numpy."""
    q = """
    SELECT sqrt(avg((e.px-g.px)*(e.px-g.px)
                  + (e.py-g.py)*(e.py-g.py)
                  + (e.pz-g.pz)*(e.pz-g.pz))) AS pos_rmse,
           sqrt(avg((e.vx-g.vx)*(e.vx-g.vx)
                  + (e.vy-g.vy)*(e.vy-g.vy)
                  + (e.vz-g.vz)*(e.vz-g.vz))) AS vel_rmse
    FROM estimates e JOIN ground_truth g
      ON e.run_id=g.run_id AND e.step=g.step
    WHERE e.run_id=? AND e.kind=?"""
    r = con.execute(q, (run_id, kind)).fetchone()
    return r["pos_rmse"], r["vel_rmse"]


def finish_run(con, run_id, **metrics):
    sets = ", ".join(f"{k}=?" for k in metrics)
    con.execute(f"UPDATE runs SET {sets} WHERE run_id=?",
                (*metrics.values(), run_id))

## 5 · Build the database from the real calibration + the simulated flight

In [ ]:
DBF = "drone.db"
for f in (DBF, DBF+"-wal", DBF+"-shm"):
    if os.path.exists(f): os.remove(f)

sim = simulate(T=60.0, dt=0.01, seed=7, sigma_imu=SIGMA_IMU, f_gps=1)
X, t = sim["X"], sim["t"]; n = len(X)
s_flow_list = [0.12 * flow_scale(X[i, 2]) for i in range(n)]

con = connect(DBF); con.create_function("sqrt", 1, math.sqrt)
run_id = new_run(con, dict(T=60.0, dt=0.01, seed=7, sigma_imu=SIGMA_IMU,
                           sigma_alt=0.60, sigma_flow=0.12, k_drag=0.8, tau=5.0,
                           sigma_gust=1.5, imu_source=IMU_LOG_URL),
                 notes="IMU noise measured from a real PX4 log")

con.execute("INSERT OR REPLACE INTO imu_calibration VALUES(?,datetime('now'),?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (IMU_LOG_URL, float(FS), float(TT[-1]-TT[0]), len(TT),
             *map(float, ACC_SIGMA), *map(float, GYR_SIGMA), *map(float, ACC_BIAS), VRW))
_st = max(1, len(TT)//4000)
con.executemany("INSERT OR REPLACE INTO imu_raw VALUES(?,?,?,?,?,?,?,?,?)",
    [(IMU_LOG_URL, i, float(TT[i]-TT[0]), *map(float, ACC[i]), *map(float, GYR[i]))
     for i in range(0, len(TT), _st)])

write_truth(con, run_id, t, X)
write_sensors(con, run_id, t, sim["imu"], sim["z_alt"], sim["z_flow"], sim["z_gps"],
              0.60, s_flow_list, 2.5)

print("rows per table:")
for tb in ("runs","imu_calibration","imu_raw","ground_truth","sensor_readings","estimates"):
    print(f"  {tb:<18}{con.execute(f'SELECT count(*) c FROM {tb}').fetchone()['c']:>7}")
print("\nsensor mix, straight from SQL:")
for r in con.execute("SELECT sensor, count(*) c, round(avg(sigma),4) s FROM sensor_readings "
                     "WHERE run_id=? GROUP BY sensor ORDER BY c DESC", (run_id,)):
    print(f"  {r['sensor']:<15}{r['c']:>6}   mean sigma {r['s']}")

## 6 · Run the filter — reading its measurements **back out of the database**

In [ ]:
imu_db, z_alt_db, z_flow_db, z_gps_db, _ = read_sensor_stream(con, run_id, n)
assert np.allclose(imu_db, sim["imu"]), "round-trip mismatch"
fed = dict(sim); fed.update(imu=imu_db, z_alt=z_alt_db, z_flow=z_flow_db, z_gps=z_gps_db)

filt      = kalman_filter(fed, adaptive_flow=True)
filt_bad  = kalman_filter(fed, adaptive_flow=False)     # for the NIS finding
sm        = rts_smoother(fed, filt)
dr        = dead_reckon(fed)

write_estimates(con, run_id, t, "filter",      filt["xf"], filt["Pf"])
write_estimates(con, run_id, t, "smoother",    sm["xs"],   sm["Ps"])
write_estimates(con, run_id, t, "dead_reckon", dr)

print("accuracy, computed by SQL JOIN against ground_truth:\n")
print(f"  {'method':<26}{'pos RMSE (m)':>14}{'vel RMSE (m/s)':>16}")
RES = {}
for kind in ("dead_reckon", "filter", "smoother"):
    p, v = rmse_by_sql(con, run_id, kind); RES[kind] = (p, v)
    print(f"  {kind:<26}{p:>14.3f}{v:>16.3f}")
print(f"\n  improvement: {RES['dead_reckon'][0]/RES['filter'][0]:.0f}x")

## 7 · Consistency — NEES and NIS (the χ² tests that catch a lying filter)

In [ ]:
def consistency(f):
    na = np.array([v for (_, g, v, _) in f["nis"] if g == "alt"])
    nf = np.array([v for (_, g, v, _) in f["nis"] if g == "flow"])
    ne = np.array([(X[i,:6]-f["xf"][i,:6]) @ np.linalg.inv(f["Pf"][i][:6,:6])
                   @ (X[i,:6]-f["xf"][i,:6]) for i in range(0, n, 10)])
    return ne, na, nf

NEES, NIS_A, NIS_F = consistency(filt)
_,    _,     NIS_F_BAD = consistency(filt_bad)
wind_rmse = rmse(filt["xf"][:, 6:], X[:, 6:])

print("FINDING 1 — NIS catches a model mismatch that RMSE hides\n")
print(f"  {'R for optical flow':<24}{'NIS (want 2)':>14}{'pos RMSE':>12}")
print(f"  {'constant (wrong)':<24}{NIS_F_BAD.mean():>14.2f}{rmse(filt_bad['xf'][:,:3],X[:,:3]):>12.3f}")
print(f"  {'altitude-scaled':<24}{NIS_F.mean():>14.2f}{rmse(filt['xf'][:,:3],X[:,:3]):>12.3f}")
print("\n  Accuracy barely moves - but the wrong filter reported uncertainty ~3x too small.")
print(f"\n  NEES {NEES.mean():.2f} (want 6)   NIS_alt {NIS_A.mean():.2f} (want 1)   NIS_flow {NIS_F.mean():.2f} (want 2)")
print(f"  wind RMSE {wind_rmse:.3f} m/s  <- never measured by any sensor")

finish_run(con, run_id, pos_rmse_dr=RES["dead_reckon"][0], pos_rmse_kf=RES["filter"][0],
           pos_rmse_rts=RES["smoother"][0], vel_rmse_kf=RES["filter"][1],
           wind_rmse_kf=wind_rmse, loglik=filt["loglik"], nees=float(NEES.mean()),
           nis_alt=float(NIS_A.mean()), nis_flow=float(NIS_F.mean()))

## 8 · All the graphs

In [ ]:
sd = lambda P, j: np.sqrt(P[:, j, j])
L  = "xyz"
fig = plt.figure(figsize=(15, 13))

ax = fig.add_subplot(3, 3, 1, projection="3d")
ax.plot(*X[:, :3].T, lw=1.8, label="true")
ax.plot(*filt["xf"][:, :3].T, lw=.9, label="Kalman")
ax.set_title("2 — 3-D trajectory"); ax.legend(fontsize=7)

ax = fig.add_subplot(3, 3, 2)
for j in range(3):
    ax.plot(t, X[:, j], lw=1.6)
    ax.plot(t, filt["xf"][:, j], lw=.7)
    ax.fill_between(t, filt["xf"][:,j]-2*sd(filt["Pf"],j), filt["xf"][:,j]+2*sd(filt["Pf"],j), alpha=.18)
ax.set(title="3 — position, true vs est $\pm2\sigma$", xlabel="s", ylabel="m")

ax = fig.add_subplot(3, 3, 3)
for j in range(3): ax.plot(t, X[:, 3+j], lw=1.6); ax.plot(t, filt["xf"][:, 3+j], lw=.7)
ax.set(title="4 — velocity, true vs estimated", xlabel="s", ylabel="m/s")

ax = fig.add_subplot(3, 3, 4)
for j in range(3):
    e = filt["xf"][:, j] - X[:, j]
    ax.plot(t, e, lw=.6, label=f"e{L[j]}")
    ax.plot(t,  2*sd(filt["Pf"], j), "k--", lw=.4); ax.plot(t, -2*sd(filt["Pf"], j), "k--", lw=.4)
ax.set(title="5 — error inside its $\pm2\sigma$ envelope", xlabel="s", ylabel="m"); ax.legend(fontsize=7)

ax = fig.add_subplot(3, 3, 5)
for j in range(3): ax.plot(t, sd(filt["Pf"], j), label=f"$\sigma_{L[j]}$")
ax.set(title="6 — x,y grow without bound; z is pinned", xlabel="s", ylabel="m"); ax.legend(fontsize=7)

ax = fig.add_subplot(3, 3, 6)
ax.plot(filt["gains"], lw=.8); ax.set(title="7 — Kalman gain (altimeter$\to z$) converges", xlabel="update #")

ax = fig.add_subplot(3, 3, 7)
for j in range(3): ax.plot(t, X[:, 6+j], lw=1.6); ax.plot(t, filt["xf"][:, 6+j], lw=.7)
ax.set(title=f"8 — wind, true vs estimated (RMSE {wind_rmse:.2f} m/s)", xlabel="s", ylabel="m/s")

ax = fig.add_subplot(3, 3, 8)
ax.plot(NIS_F_BAD, lw=.3, alpha=.6, label=f"constant R  (mean {NIS_F_BAD.mean():.1f})")
ax.plot(NIS_F,     lw=.3, alpha=.8, label=f"altitude-scaled  (mean {NIS_F.mean():.1f})")
ax.axhline(2, color="k", lw=1, ls="--", label="target = 2")
ax.set(title="9 — NIS exposes the overconfident filter", xlabel="flow update #", ylim=(0, 18)); ax.legend(fontsize=6)

ax = fig.add_subplot(3, 3, 9)
names = ["IMU only", "Kalman", "Smoother"]
vals  = [RES["dead_reckon"][0], RES["filter"][0], RES["smoother"][0]]
b = ax.bar(names, vals, color=["#6A7179", "#E07A3F", "#2E7FA8"])
ax.bar_label(b, fmt="%.2f m", fontsize=8); ax.set_yscale("log")
ax.set(title="10 — position RMSE (log scale)", ylabel="m")

plt.suptitle("Drone state estimation — LDS + Kalman filter + RTS smoother", y=1.0, fontsize=13)
plt.tight_layout(); plt.show()

## 9 · EM — learning the sensor noise from the data alone

E‑step = filter + smoother (the posterior over hidden states). M‑step = set each variance to the
mean squared smoothed residual. We start **3× too high** on purpose.

In [ ]:
R_alt_hat, R_flow_hat, hist = em_learn_noise(fed, iters=8)
ll = [h[1] for h in hist]
print(f"\n  recovered sigma_alt = {np.sqrt(R_alt_hat[0,0]):.3f}  (true {fed['sigma_alt']:.3f})")
print(f"  log-likelihood {ll[0]:.1f} -> {ll[-1]:.1f}, monotonic: {all(b>=a-1e-6 for a,b in zip(ll,ll[1:]))}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(ll, "o-"); ax[0].set(title="11 — EM log-likelihood rises monotonically", xlabel="iteration")
ax[1].plot([h[2] for h in hist], "o-", label="$\\sigma_{alt}$ estimate")
ax[1].axhline(fed["sigma_alt"], color="k", ls="--", label="truth")
ax[1].set(title="12 — EM recovers the true noise", xlabel="iteration"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 10 · Finding 2 — when is a noisy GPS actually worth adding?

Horizontal position is **unobservable** from IMU + altimeter + flow, so its variance grows
forever. A 2.5 m GPS only helps once that drift exceeds its own noise.

In [ ]:
print(f"  {'flight':<9}{'no GPS':>11}{'with GPS':>11}   verdict")
for T in (60, 120, 300, 600):
    sg = simulate(T=float(T), dt=0.01, seed=7, sigma_imu=SIGMA_IMU, f_gps=1)
    a = kalman_filter(sg, adaptive_flow=True)
    b = kalman_filter(sg, adaptive_flow=True, use_gps=True)
    ra = rmse(a["xf"][:,:3], sg["X"][:,:3]); rb = rmse(b["xf"][:,:3], sg["X"][:,:3])
    print(f"  {T:>4d} s   {ra:>9.3f} m {rb:>9.3f} m   "
          f"{'GPS HURTS' if rb>ra else 'GPS helps '} ({100*(rb-ra)/ra:+.0f}%)")

## 11 · Export the database

In [ ]:
con.execute("PRAGMA wal_checkpoint(TRUNCATE)")
print("runs table:")
for k, v in dict(con.execute("SELECT * FROM runs WHERE run_id=?", (run_id,)).fetchone()).items():
    if k != "params": print(f"  {k:<15}{round(v,4) if isinstance(v,float) else v}")

with open("drone_db.sql", "w") as f:
    for line in con.iterdump(): f.write(line + "\n")
con.close()
print(f"\n  drone.db       {os.path.getsize('drone.db')/1024:.0f} KB")
print(f"  drone_db.sql   {os.path.getsize('drone_db.sql')/1024:.0f} KB  (portable dump)")
try:
    from google.colab import files
    files.download("drone_db.sql")
except Exception:
    pass

---
## Conclusions

| | |
|---|---|
| IMU only (dead reckoning) | **53.4 m** |
| Kalman filter (real time) | **0.35 m** — 155× better |
| RTS smoother (offline) | **0.24 m**, velocity 3.1× better |
| Wind, never measured | recovered to **1.22 m/s** |

**What we'd stress.** NIS caught a filter that was accurate but 3× overconfident — RMSE alone
would have missed it. And horizontal position is genuinely unobservable here, which is *why*
real drones carry GPS.

**Honest limits.** The trajectory is simulated; only the IMU noise is real. The altimeter and
optical‑flow streams are derived from the trajectory, not raw hardware. NEES of 4.15 vs a target
of 6 means we stay mildly conservative on the wind states.

**Next.** The real log showed a 1.09 m/s² accelerometer bias — the obvious next move is to add
bias as extra states.